# Week 11 - Applied Mini Project

Project: Supermarket Sales Prediction

In this project I explore the supermarket sales data and then use linear regression to predict the sales of an order.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Business question

In [ ]:
print("Business Question:")
print("Can we predict the sales of an order using information such as")
print("shipping mode, customer segment, region, category, sub-category")
print("and order date?")

## 3. Load data

In [ ]:
data = pd.read_csv("supermarket.csv")

print("\nFirst 5 rows:")
print(data.head())

print("\nShape of dataset:")
print(data.shape)

print("\nColumn names:")
print(data.columns)

print("\nDataset information:")
print(data.info())

## 4. Check missing values

In [ ]:
print("\nMissing values:")
print(data.isnull().sum())

## 5. Check duplicates

In [ ]:
print("\nNumber of duplicate rows:")
print(data.duplicated().sum())

## 6. Basic statistics

In [ ]:
print("\nBasic statistics:")
print(data.describe())

## 7. Convert date columns

In [ ]:
data["Order Date"] = pd.to_datetime(data["Order Date"])
data["Ship Date"] = pd.to_datetime(data["Ship Date"])

print("\nDate columns converted successfully.")

## 8. Create shipping days

Shipping days is the ship date minus the order date.

In [ ]:
data["Shipping Days"] = (
    data["Ship Date"] - data["Order Date"]
).dt.days

print("\nShipping Days:")
print(data["Shipping Days"].head())

## 9. Create date features

In [ ]:
data["Order Year"] = data["Order Date"].dt.year
data["Order Month"] = data["Order Date"].dt.month

print("\nNew date features:")
print(data[["Order Date", "Order Year", "Order Month"]].head())

## 10. Sales distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data["Sales"], bins=30)
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Number of Orders")
plt.show()

## 11. Total sales

In [ ]:
total_sales = data["Sales"].sum()

print("\nTotal Sales:")
print(round(total_sales, 2))

## 12. Average sales

In [ ]:
average_sales = data["Sales"].mean()

print("\nAverage Sales per Order:")
print(round(average_sales, 2))

## 13. Sales by category

In [ ]:
category_sales = data.groupby("Category")["Sales"].sum().sort_values(ascending=False)

print("\nSales by Category:")
print(category_sales)

In [ ]:
plt.figure(figsize=(8, 5))
category_sales.plot(kind="bar")
plt.title("Sales by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.show()

## 14. Sales by region

In [ ]:
region_sales = data.groupby("Region")["Sales"].sum().sort_values(ascending=False)

print("\nSales by Region:")
print(region_sales)

In [ ]:
plt.figure(figsize=(8, 5))
region_sales.plot(kind="bar")
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.show()

## 15. Sales by customer segment

In [ ]:
segment_sales = data.groupby("Segment")["Sales"].sum().sort_values(ascending=False)

print("\nSales by Customer Segment:")
print(segment_sales)

In [ ]:
plt.figure(figsize=(8, 5))
segment_sales.plot(kind="bar")
plt.title("Sales by Customer Segment")
plt.xlabel("Segment")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.show()

## 16. Sales by sub-category

In [ ]:
subcategory_sales = (
    data.groupby("Sub-Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 10 Sub-Categories by Sales:")
print(subcategory_sales.head(10))

In [ ]:
plt.figure(figsize=(10, 5))
subcategory_sales.head(10).plot(kind="bar")
plt.title("Top 10 Sub-Categories by Sales")
plt.xlabel("Sub-Category")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.show()

## 17. Monthly sales trend

In [ ]:
monthly_sales = data.groupby(
    ["Order Year", "Order Month"]
)["Sales"].sum()

print("\nMonthly Sales:")
print(monthly_sales)

In [ ]:
# Create a proper date for plotting
data["Month Date"] = data["Order Date"].dt.to_period("M").dt.to_timestamp()

monthly_sales_plot = data.groupby("Month Date")["Sales"].sum()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales_plot.index, monthly_sales_plot.values)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.show()

## 18. Check sales by year

In [ ]:
year_sales = data.groupby("Order Year")["Sales"].sum()

print("\nSales by Year:")
print(year_sales)

In [ ]:
plt.figure(figsize=(8, 5))
year_sales.plot(kind="bar")
plt.title("Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)
plt.show()

## 19. Prepare data for machine learning

I only keep the useful columns. IDs and names are not used for the model.

In [ ]:
# We will predict Sales

# Columns such as IDs and names are not useful for this model.
# They are mainly identifiers rather than useful business features.

model_data = data[
    [
        "Ship Mode",
        "Segment",
        "Region",
        "Category",
        "Sub-Category",
        "Order Year",
        "Order Month",
        "Shipping Days",
        "Sales"
    ]
].copy()

print("\nData used for machine learning:")
print(model_data.head())

## 20. Check missing values again

In [ ]:
print("\nMissing values in model data:")
print(model_data.isnull().sum())

## 21. Convert categorical variables

Machine learning models need numbers, so the text columns are converted with get_dummies.

In [ ]:
model_data = pd.get_dummies(
    model_data,
    columns=[
        "Ship Mode",
        "Segment",
        "Region",
        "Category",
        "Sub-Category"
    ],
    drop_first=True
)

print("\nData after encoding:")
print(model_data.head())

## 22. Separate features and target

In [ ]:
X = model_data.drop("Sales", axis=1)

y = model_data["Sales"]

print("\nNumber of features:")
print(X.shape[1])

## 23. Train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining data size:")
print(X_train.shape)

print("\nTesting data size:")
print(X_test.shape)

## 24. Train linear regression model

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

print("\nLinear Regression model trained successfully.")

## 25. Make predictions

In [ ]:
y_pred = model.predict(X_test)

print("\nFirst 10 predictions:")
print(y_pred[:10])

## 26. Model evaluation

The model is checked using MAE, MSE, RMSE and R2 score.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

In [ ]:
print("\nMODEL EVALUATION")
print("-------------------------")

print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

## 27. Actual vs predicted sales

In [ ]:
comparison = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred
})

print("\nActual vs Predicted Sales:")
print(comparison.head(10))

## 28. Actual vs predicted graph

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(y_test, y_pred)

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")

plt.show()

## 29. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

feature_importance["Absolute Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

feature_importance = feature_importance.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

print("\nMost important features:")
print(feature_importance.head(10))

## 30. Plot important features

In [ ]:
top_features = feature_importance.head(10)

plt.figure(figsize=(10, 5))

plt.bar(
    top_features["Feature"],
    top_features["Absolute Coefficient"]
)

plt.title("Top 10 Important Features")
plt.xlabel("Features")
plt.ylabel("Absolute Coefficient")

plt.xticks(rotation=75)

plt.show()

## 31. Business findings

In [ ]:
print("\nBUSINESS FINDINGS")
print("-------------------------")

print("1. The dataset contains order, customer, product and sales information.")

print("2. Sales can be compared across different categories, regions and customer segments.")

print("3. The monthly sales graph helps identify changes in sales over time.")

print("4. Category and sub-category analysis shows which product groups generate more sales.")

print("5. A Linear Regression model was used to predict sales using order-related features.")

print("6. The model performance can be checked using MAE, RMSE and R2 score.")

## 32. Business implication

In [ ]:
print("\nBUSINESS IMPLICATION")
print("-------------------------")

print("The analysis can help a business understand sales patterns")
print("across regions, customer segments and product categories.")

print("Sales trends can also help businesses plan inventory and")
print("focus attention on product categories with higher sales.")

print("The prediction model can provide an estimated sales value")
print("based on the information available for an order.")

print("However, the dataset does not contain some useful variables")
print("such as quantity, discount and profit, so adding these variables")
print("could improve future sales prediction.")

## 33. Final conclusion

In [ ]:
print("\nFINAL CONCLUSION")
print("-------------------------")

print("This project analyzed a supermarket sales dataset and")
print("used machine learning to predict sales.")

print("The project included data cleaning, exploratory data analysis,")
print("feature engineering, visualization and Linear Regression.")

print("The results show how historical sales data can be used to")
print("understand business patterns and support sales-related decisions.")

## 34. Mentor feedback

In [ ]:
print("\nMENTOR FEEDBACK")
print("-------------------------")

print("Mentor feedback can be added here after discussing the project.")